# Feature Engineering
In this file, feature engineering is performed, and a couple new features are added to the data, including:
* ***PropertyAge*** —> how old the property is 
* ***BedBathRatio*** -> Ratio of bedrooms to bathrooms in the property
* ***AmenityScore*** -> A number indicating how many amenities the property has
* ***SchoolDistrict_encoded*** -> the target encoded CA school district the property lies in

### Import Libraries and Load Data

In [1]:
import pandas as pd
import geopandas as gpd

from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, median_absolute_error

In [2]:
# sets for linear regression
linear_train = pd.read_csv('data/model_sets/train_set_scaled.csv')
linear_test = pd.read_csv('data/model_sets/test_set_scaled.csv')

# sets for tree models
tree_train = pd.read_csv('data/model_sets/train_set_unscaled.csv')
tree_test = pd.read_csv('data/model_sets/test_set_unscaled.csv')

In [3]:
linear_train.columns

Index(['ViewYN', 'WaterfrontYN', 'BasementYN', 'PoolPrivateYN', 'Latitude',
       'Longitude', 'LivingArea', 'CountyOrParish', 'AttachedGarageYN',
       'ParkingTotal', 'YearBuilt', 'BathroomsTotalInteger', 'City',
       'BedroomsTotal', 'FireplaceYN', 'Stories', 'Levels',
       'MainLevelBedrooms', 'NewConstructionYN', 'GarageSpaces',
       'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'logClosePrice'],
      dtype='object')

### Initial Feature Engineering

In [4]:
currentYear = 2026 
amenities = [col for col in linear_train.columns if col.endswith('YN')]
for df in [linear_train, linear_test, tree_test, tree_train]:
    # PropertyAge — years since the property was first built
    df['PropertyAge'] = 2026 - df['YearBuilt']

    # BedBathRatio — ratio of bedrooms to bathrooms in property
    df['BedBathRatio'] = df['BathroomsTotalInteger'] / df['BedroomsTotal'].replace(0,1)

    # Amenity Score
    for col in amenities:
        df[col] = df[col].astype(int)
    df['AmenityScore'] = df[amenities].sum(axis=1)

### School District Geospatial Feature

In [5]:
# read GeoJSON
districts = gpd.read_file('data/enriched_sets/DistrictAreas2526_-284845464123469011.geojson')

# Create GeoDataFrame from property coordinates
def spatial_join_districts(prop_df, districts_gdf): 
    """Spatially join properties to school district boundaries.
    Prioritizes Unified districts over Elementary/High."""
    
    # Convert each house into a point on a map
    prop_gdf = gpd.GeoDataFrame(
        prop_df,
        geometry=gpd.points_from_xy(prop_df['Longitude'], prop_df['Latitude']),
        crs='EPSG:4326'
    )
    
    # Make sure both datasets use the same coordinate system
    if districts_gdf.crs != prop_gdf.crs:
        districts_gdf = districts_gdf.to_crs(prop_gdf.crs)
    
    # Keep only relevant columns from districts
    dist_cols = ['DistrictNa', 'DistrictTy', 'geometry']

    # Handle column name variations
    for alt_name, std_name in [('DistrictName', 'DistrictNa'), ('DistrictType', 'DistrictTy')]:
        if alt_name in districts_gdf.columns and std_name not in districts_gdf.columns:
            districts_gdf = districts_gdf.rename(columns={alt_name: std_name})
    
    available_cols = [c for c in dist_cols if c in districts_gdf.columns]
    
    # Perform spatial join
    joined = gpd.sjoin(
        prop_gdf,
        districts_gdf[available_cols],
        how='left', # keep every property, even if it doesn't fall inside a district
        predicate='within' # match the property only if the point lies inside the district polygon
    )
    
    # Handle multiple matches (overlapping districts)
    # Priority: Unified > Elementary > High > any other
    if 'DistrictTy' in joined.columns:
        priority_map = {'Unified': 0, 'Elementary': 1, 'High': 2}
        joined['_priority'] = joined['DistrictTy'].map(priority_map).fillna(3)
        
        # Keep highest-priority match per property
        joined = joined.sort_values('_priority').drop_duplicates(
            subset=[c for c in prop_df.columns if c != 'geometry'],
            keep='first'
        )
        joined = joined.drop(columns=['_priority'], errors='ignore')
    else:
        # If no district type column, just keep first match
        joined = joined.drop_duplicates(
            subset=[c for c in prop_df.columns if c != 'geometry'],
            keep='first'
        )
    
    # Drop GeoDataFrame-specific columns
    result = pd.DataFrame(joined.drop(columns=['geometry', 'index_right'], errors='ignore'))
    
    return result

In [6]:
datasets = {
    "linear_train": linear_train,
    "linear_test": linear_test,
    "tree_train": tree_train,
    "tree_test": tree_test
}

# perform spatial join on each split 
for name, df in datasets.items():
    print(f"Spatial joining {name}...")
    
    # Perform spatial join
    datasets[name] = spatial_join_districts(df, districts)
    
    # Report matches
    matched = datasets[name]["DistrictNa"].notna().sum()
    total = len(datasets[name])
    print(f"  Matched: {matched:,} / {total:,} properties")

    # Fill unmatched districts
    datasets[name]["DistrictNa"] = datasets[name]["DistrictNa"].fillna("Unknown")

# Unpack back into the original variables
linear_train = datasets["linear_train"]
linear_test  = datasets["linear_test"]
tree_train   = datasets["tree_train"]
tree_test    = datasets["tree_test"]

# drop DistrictsTy from the datasets 
linear_train = linear_train.drop(columns=['DistrictTy'])
linear_test = linear_test.drop(columns=['DistrictTy'])
tree_train = tree_train.drop(columns=['DistrictTy'])
tree_test = tree_test.drop(columns=['DistrictTy'])

# fill remaining missing values with "Unknown"
linear_train['DistrictNa'] = linear_train['DistrictNa'].fillna('Unknown')
linear_test['DistrictNa'] = linear_test['DistrictNa'].fillna('Unknown')
tree_train['DistrictNa'] = tree_train['DistrictNa'].fillna('Unknown')
tree_test['DistrictNa'] = tree_test['DistrictNa'].fillna('Unknown')

for name, data in datasets.items():
    print(f"Unique districts in {name}: {data['DistrictNa'].nunique()}")

Spatial joining linear_train...
  Matched: 21,263 / 21,270 properties
Spatial joining linear_test...
  Matched: 10,979 / 10,984 properties
Spatial joining tree_train...
  Matched: 21,263 / 21,270 properties
Spatial joining tree_test...
  Matched: 10,979 / 10,984 properties
Unique districts in linear_train: 501
Unique districts in linear_test: 470
Unique districts in tree_train: 501
Unique districts in tree_test: 470


### Target Encoding `DistrictNa` 
Since `DistrictNa` is a categorical column, it will be target encoded using bayesian smoothing—similar to how it was done in `02_preprocessing.ipynb`. Encodings will be calculated on training set and applied on test set to avoid leakage.

In [7]:
def target_encode(train_df, test_df, target='logClosePrice', smoothing=50):
    """Target-encode a categorical column using training data only."""
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    stats['weight'] = stats['count'] / (stats['count'] + smoothing)
    stats['encoded'] = stats['weight'] * stats['mean'] + (1 - stats['weight']) * global_mean
    encoding_map = stats['encoded'].to_dict()
    
    train_encoded = train_df[col].map(encoding_map).fillna(global_mean)
    test_encoded = test_df[col].map(encoding_map).fillna(global_mean)
    return train_encoded, test_encoded, encoding_map 

# target encode linear splits
linear_train["DistrictNa"], linear_test["DistrictNa"], district_map = target_encode(
    linear_train,
    linear_test
)

# target encode tree splits
tree_train["DistrictNa"], tree_test["DistrictNa"], district_map_tree = target_encode(
    tree_train,
    tree_test
)

for name, data in datasets.items():
    print(f'{name} columns: {len(data.columns)}')

linear_train columns: 30
linear_test columns: 30
tree_train columns: 30
tree_test columns: 30


### Model Training and Evaluation
Linear, Decision Tree, and Random Forest models will be retrained and evaluated on the new feature sets.

In [8]:
# initialize models
linear = LinearRegression()
dt = DecisionTreeRegressor(random_state=54)
rf = RandomForestRegressor(n_estimators=100, random_state=54)

# linear splits
X_train_lin = linear_train.drop(columns=['logClosePrice'])
y_train_lin = linear_train['logClosePrice']
X_test_lin = linear_test.drop(columns=['logClosePrice'])
y_test_lin = linear_test['logClosePrice']

# tree splits
X_train_tree = tree_train.drop(columns=['logClosePrice'])
y_train_tree = tree_train['logClosePrice']
X_test_tree = tree_test.drop(columns=['logClosePrice'])
y_test_tree = tree_test['logClosePrice']

In [11]:
# fit models
linear.fit(X_train_lin, y_train_lin)
dt.fit(X_train_tree, y_train_tree)
rf.fit(X_train_tree, y_train_tree)

# make predictions
linear_preds = linear.predict(X_test_lin)
dt_preds = dt.predict(X_test_tree)
rf_preds = rf.predict(X_test_tree)

# calculate metrics
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest'],
    'R2': [
        r2_score(y_test_lin, linear_preds),
        r2_score(y_test_tree, dt_preds),
        r2_score(y_test_tree, rf_preds)
    ],
    'MAE': [
        mean_absolute_error(y_test_lin, linear_preds),
        mean_absolute_error(y_test_tree, dt_preds),
        mean_absolute_error(y_test_tree, rf_preds)
    ],
    'MdAE': [
        median_absolute_error(y_test_lin, linear_preds),
        median_absolute_error(y_test_tree, dt_preds),
        median_absolute_error(y_test_tree, rf_preds)
    ]
})

# round metrics
results[['R2', 'MAE', 'MdAE']] = results[['R2', 'MAE', 'MdAE']].round(4)

results

,Model,R2,MAE,MdAE
0,Linear Regression,0.7812,0.1943,0.1456
1,Decision Tree,0.7801,0.1828,0.1214
2,Random Forest,0.8766,0.1322,0.0882


### Summary
| **Model** | **R2 score** | **MAE** | **MdAE**
|---|---|---|---|
|*Linear*| 0.7812 | 0.1943 | 0.1456 |
|*Decision Tree*| 0.7801 | 0.1828 | 0.1214 |
|*Random Forest*| 0.8766| 0.1322 | 0.0882 |

* Decision tree model performed slightly better with the added features, while Linear and Random Forest had little significant change
* As a whole, Random Forest Model performs best in all 3 metrics, which follows expectation